In [1]:
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

0.3.27
0.3.24


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor 
from langchain.prompts import ChatPromptTemplate

import utils

# Create model
llm = ChatOpenAI(model="gpt-4o-mini")

c:\study\AI\IITM_Agentic_AI_Training\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Mock API Tool: Check Claim Status
@tool
def check_claim_status(claim_id: str) -> str:
    """Return status of a claim with given claim_id."""
    fake_db = {
        "C101": "Approved - Payment in process",
        "C202": "Under review - Waiting documents",
        "C303": "Flagged for manual investigation"
    }
    return fake_db.get(claim_id, "Claim ID not found")

# Fraud Detection Tool
@tool
def fraud_risk_check(description: str) -> str:
    """Returns risk evaluation for fraud based on description."""
    if "lost phone but still using it" in description.lower():
        return "High Risk"
    return "Low Risk"

# Escalation Tool
@tool
def escalate_to_human(issue: str) -> str:
    """Escalates conversation to a human agent."""
    return f"Escalation created. A human agent will review: {issue}"

In [4]:
tools = [check_claim_status, fraud_risk_check, escalate_to_human]

SYSTEM_PROMPT = """
You are an Insurance Claims Assistant.

Your job:
1) Understand user intent.
2) If they ask about a claim status → call `check_claim_status`
3) If user mentions incident details → call `fraud_risk_check`
4) If unsure, conflicting, or user upset → call `escalate_to_human`

Always respond in a friendly, formal tone.
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

In [5]:
from openai import max_retries


agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)
executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True)

In [6]:
query = "What's the current status of claim C202?"
result = executor.invoke({"input": query})
print("Agent Output:", result)



> Entering new AgentExecutor chain...

Invoking: `check_claim_status` with `{'claim_id': 'C202'}`


Under review - Waiting documentsThe current status of claim C202 is "Under review - Waiting for documents." If you need further assistance or updates, please let me know!

> Finished chain.
Agent Output: {'input': "What's the current status of claim C202?", 'output': 'The current status of claim C202 is "Under review - Waiting for documents." If you need further assistance or updates, please let me know!'}


In [7]:
query = "I lost my phone but it looks like it is being used by someone."
result = executor.invoke({"input": query})
print("Agent Output:", result)



> Entering new AgentExecutor chain...

Invoking: `escalate_to_human` with `{'issue': 'User lost their phone and suspects it is being used by someone else.'}`


Escalation created. A human agent will review: User lost their phone and suspects it is being used by someone else.I understand that losing your phone and suspecting it's being used by someone else can be very distressing. I've escalated this issue to a human agent who will assist you further. Thank you for your patience.

> Finished chain.
Agent Output: {'input': 'I lost my phone but it looks like it is being used by someone.', 'output': "I understand that losing your phone and suspecting it's being used by someone else can be very distressing. I've escalated this issue to a human agent who will assist you further. Thank you for your patience."}
